In [21]:
import unicodedata
import numpy as np
import pandas as pd
from keras.src.utils.module_utils import tensorflow
from tensorflow.keras.utils import Sequence
from tensorflow.keras.layers import Conv2D,Dense,Dropout,Input,LSTM,Embedding,MultiHeadAttention,LayerNormalization
import os
from tensorflow.keras.callbacks import EarlyStopping
import datasets
from datasets import Dataset,DatasetDict
import tensorflow as tf
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [22]:
with open ("D:/project_deeplearning/TEP.en-fa.en",encoding="utf-8") as f:
    en_s=f.read().splitlines()

with open ("D:/project_deeplearning/TEP.en-fa.fa",encoding="utf-8") as f:
    fa_s=f.read().splitlines()


assert len(en_s)==len(fa_s)

df=pd.DataFrame(
    {
        "en":en_s,
        "fa":fa_s
    }
)

df=df.sample(n=30000,random_state=42)
df=df.reset_index(drop=True)

en=df["en"]
fa=df["fa"]
data=pd.DataFrame({"train":Dataset.from_pandas(df)})
print(data["train"][0])

{'en': 'stop her . somebody stop her reading .', 'fa': 'متوقفش كنيد يک نفر نگذاره اون ادامه بده .'}


In [23]:
train=data["train"]
train[:10]

0    {'en': 'stop her . somebody stop her reading ....
1        {'en': 'tetrastichous .', 'fa': 'چهاربيتي .'}
2    {'en': 'thats her stage name . i just said tha...
3    {'en': 'i wanna go . what .', 'fa': 'ميخوام بر...
4    {'en': 'im going to see the dragon warrior .',...
5    {'en': 'just think that i've received them and...
6    {'en': 'and the food was no different from a p...
7    {'en': 'are a couple jerkoffs .', 'fa': '2تا ا...
8    {'en': 'i think i should probably just stay wi...
9                {'en': 'vail .', 'fa': 'بکارخوردن .'}
Name: train, dtype: object

In [24]:
def unicode_to_asci(s):
    return "".join(c for c in unicodedata.normalize("NFC",s) if unicodedata.category(c)!='MN' )

In [25]:
len(data)

30000

In [26]:
def preprossing(w):
    w=unicode_to_asci(w.lower().strip())
    w=re.sub(r"([.!?])",r"\1",w)
    w=re.sub(r'([""])+',"",w)
    w=w.rstrip().strip()
    w = "<start> " + w + " <end>"
    return w

In [27]:
en_sen="Im very happy."
preprossing(en_sen)

'<start> im very happy. <end>'

In [28]:
fa_sen="درود بر تو."
preprossing(fa_sen)

'<start> درود بر تو. <end>'

In [29]:
df["en"]=df["en"].apply(preprossing)
df["fa"]=df["fa"].apply(preprossing)

In [30]:
vocab_size=15000
max_length=50
batch_size=32

token_en=Tokenizer(num_words=vocab_size,filters="",oov_token="<unk>")
token_fa=Tokenizer(num_words=vocab_size,filters="",oov_token="<unk>")



token_en.fit_on_texts(df["en"])
token_fa.fit_on_texts(df["fa"])

en_seq=token_en.texts_to_sequences(df["en"])
fa_seq=token_fa.texts_to_sequences(df["fa"])

en_seq=pad_sequences(en_seq , maxlen=max_length,padding="post")
fa_seq=pad_sequences(fa_seq,maxlen=max_length,padding="post")

decoder_inputs_array=fa_seq[:,:-1]
decoder_targets_array=fa_seq[:,1:]




In [31]:
latent_dim=256

encoder_inputs=Input(shape=(max_length,),name="encoder_inputs")
encoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(latent_dim,return_state=True,return_sequences=True,dropout=0.3,recurrent_dropout=0.1)(encoder_embedding)



decoder_inputs=Input(shape=(None,),name="decoder_inputs")
decoder_embedding=Embedding(input_dim=vocab_size,output_dim=latent_dim,mask_zero=True,name="decoder_embedding")
decoder_embedd=decoder_embedding(decoder_inputs)
decoder_lstm=LSTM(latent_dim,return_sequences=True,return_state=True,name="decoder_lstm",dropout=0.5,recurrent_dropout=0.1)
decoder_outputs,state_h_dec,state_c_dec=decoder_lstm(decoder_embedd,initial_state=[state_h,state_c])

atn=tf.keras.layers.Attention(name='simple_attention')
atn_out=atn([decoder_outputs,encoder_output])
concat_out=tf.keras.layers.Concatenate(name="concat_attention")([decoder_outputs,atn_out])

In [32]:
decoder_dense=Dense(vocab_size,activation="softmax",kernel_regularizer=tf.keras.regularizers.l2(1e-3))
decoder_outputs=decoder_dense(concat_out)

In [33]:
model=tf.keras.Model([encoder_inputs,decoder_inputs],decoder_outputs)

In [34]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 50, 256)   │  3,840,000 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 50)        │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 256) │  3,840,000 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 50, 256), │    525,312 │ embedding_1[0][0… │
│                     │ (None, 256),      │            │ not_equal_2[0][0] │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │    525,312 │ decoder_embeddin… │
│                     │ 256), (None,      │            │ lstm_1[0][1],     │
│                     │ 256), (None,      │            │ lstm_1[0][2]      │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_attention    │ (None, None, 256) │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_attention    │ (None, None, 512) │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ simple_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None,      │  7,695,000 │ concat_attention… │
│                     │ 15000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,425,624 (62.66 MB)

 Trainable params: 16,425,624 (62.66 MB)

 Non-trainable params: 0 (0.00 B)

In [35]:
early=EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

In [36]:
splt=int(len(en_seq)*0.9)
train_ds=tf.data.Dataset.from_tensor_slices(((en_seq[:splt],decoder_inputs_array[:splt]),decoder_targets_array[:splt])).shuffle(10000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds=tf.data.Dataset.from_tensor_slices(((en_seq[splt:],decoder_inputs_array[splt:]),decoder_targets_array[splt:])).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [37]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),metrics=["accuracy"],loss="sparse_categorical_crossentropy")

In [38]:
history=model.fit(train_ds,epochs=40,validation_data=val_ds,callbacks=[early],verbose=2)

Epoch 1/40
844/844 - 622s - 737ms/step - accuracy: 0.2697 - loss: 5.7156 - val_accuracy: 0.3365 - val_loss: 5.1060
Epoch 2/40
844/844 - 513s - 608ms/step - accuracy: 0.3294 - loss: 5.1948 - val_accuracy: 0.3431 - val_loss: 5.0093
Epoch 3/40
844/844 - 650s - 770ms/step - accuracy: 0.3353 - loss: 5.1135 - val_accuracy: 0.3468 - val_loss: 4.9508
Epoch 4/40
844/844 - 658s - 780ms/step - accuracy: 0.3425 - loss: 5.0478 - val_accuracy: 0.3527 - val_loss: 4.8986
Epoch 5/40
844/844 - 646s - 766ms/step - accuracy: 0.3481 - loss: 4.9847 - val_accuracy: 0.3571 - val_loss: 4.8595
Epoch 6/40
844/844 - 679s - 805ms/step - accuracy: 0.3547 - loss: 4.9295 - val_accuracy: 0.3600 - val_loss: 4.8238
Epoch 7/40
844/844 - 630s - 747ms/step - accuracy: 0.3602 - loss: 4.8788 - val_accuracy: 0.3643 - val_loss: 4.7948
Epoch 8/40
844/844 - 586s - 694ms/step - accuracy: 0.3667 - loss: 4.8332 - val_accuracy: 0.3675 - val_loss: 4.7735
Epoch 9/40
844/844 - 533s - 631ms/step - accuracy: 0.3728 - loss: 4.7909 - val_a

In [41]:
reverse_fa = {v: k for k, v in token_fa.word_index.items()}

In [42]:
import pickle
model.save('Translator.keras')

In [43]:
from tensorflow.keras.models import load_model
model=load_model("Translator.keras")
pickle.dump(token_fa,open("token_fa.pkl","wb"))
pickle.dump(token_en,open("token_en.pkl","wb"))
pickle.dump(reverse_fa,open("reverse_fa.pkl","wb"))

In [44]:
import pickle

token_fa=pickle.load(open("token_fa.pkl","rb"))
token_en=pickle.load(open("token_en.pkl","rb"))
reverse_fa=pickle.load(open("reverse_fa.pkl","rb"))


In [45]:
encoder_model=tf.keras.Model(encoder_inputs,[encoder_output,state_h,state_c])

In [46]:
decoder_state_input_h=Input(shape=(latent_dim,))
decoder_state_input_c=Input(shape=(latent_dim,))
enc_out_input=Input(shape=(max_length,latent_dim))
decoder_states_inputs=[decoder_state_input_h, decoder_state_input_c]

decoder_emb2=decoder_embedding(decoder_inputs)

decoder_outputs2, state_h2, state_c2 = decoder_lstm(
    decoder_emb2,
    initial_state=decoder_states_inputs
)

atn2=atn([decoder_outputs2,enc_out_input])
concat_out2=tf.keras.layers.Concatenate()([decoder_outputs2,atn2])
decoder_outputs2=decoder_dense(concat_out2)

decoder_model=tf.keras.Model(
    [decoder_inputs,decoder_state_input_h,decoder_state_input_c,enc_out_input],
    [decoder_outputs2, state_h2, state_c2]
)



In [47]:
encoder_model.save('encoder_model.keras')
decoder_model.save('decoder_model.keras')

In [48]:
def translate(sentence):

    sentence=preprossing(sentence)

    seq=token_en.texts_to_sequences([sentence])
    seq=pad_sequences(seq, maxlen=max_length, padding="post")

    enc_out,h,c=encoder_model.predict(seq)
    target_seq=np.array([[token_fa.word_index["<start>"]]])

    stop=False
    decoded= ""

    while not stop:

        output_tokens, h, c=decoder_model.predict([target_seq,h,c,enc_out])
        sampled_token_index=np.argmax(output_tokens[0, -1, :])
        sampled_word=reverse_fa.get(sampled_token_index, "")

        if sampled_word== "<end>" or len(decoded.split())>max_length:
            stop=True
        else:
            decoded+= " " +sampled_word

        target_seq=np.array([[sampled_token_index]])
        states=[h, c]

    return decoded


In [49]:
print(translate("I love you"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 386ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
 من مي تونم


In [50]:
print(translate('you can'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
 تو مي مي کني


In [ ]:
import sacrebleu

In [ ]:
print(tf.config.list_physical_devices('GPU'))

In [ ]:
print(token_fa.index_word[1])
print(token_fa.index_word[2])
print(token_fa.index_word[3])
print(token_fa.index_word[4])
print(token_fa.index_word[5])

In [20]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[]
